# Pandas II — Limpieza de datos y construcción de una tabla analítica

## Objetivos
- Diagnosticar rápidamente calidad de datos (tipos, nulos, duplicados).
- Aplicar estrategias de limpieza (nulos, duplicados, texto, tipos).
- Entender y utilizar `merge` para construir una tabla analítica.
- Crear métricas y features básicas listas para análisis.

## Entregable al final de la sesión
- `customers_clean`, `products_clean`, `transactions_clean`
- `fact_sales` (tabla analítica a nivel transacción) con `amount`


---
## 0. Setup

In [2]:
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

np.random.seed(42)
print('pandas:', pd.__version__)

pandas: 2.3.3


---
## 1. Datos de trabajo

Trabajaremos con tres tablas con problemas realistas:
- `customers`: dimensión de clientes (puede tener duplicados y nulos)
- `products`: dimensión de productos (categorías sucias, duplicados)
- `transactions`: hechos (nulos en claves, descuentos faltantes, cantidades 0, duplicados)


In [3]:
# --- Customers ---
n_customers = 120
customer_ids = np.arange(1000, 1000 + n_customers)

cities = np.random.choice(
    ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao', 'Zaragoza', None],
    size=n_customers,
    p=[0.26, 0.20, 0.16, 0.11, 0.10, 0.10, 0.07]
)
segments = np.random.choice(['Basic', 'Plus', 'Premium'], size=n_customers, p=[0.55, 0.30, 0.15])

ages = np.random.normal(loc=36, scale=12, size=n_customers).round().astype(int)
ages = np.clip(ages, 18, 75).astype(float)

customers = pd.DataFrame({
    'customer_id': customer_ids,
    'name': [f" Client_{i:03d} " for i in range(1, n_customers + 1)],
    'age': ages,
    'city': cities,
    'segment': segments
})
# Inject missing ages
customers.loc[np.random.choice(customers.index, size=10, replace=False), 'age'] = np.nan

# Dirty names: casing + extra spaces
ix_dirty = np.random.choice(customers.index, size=18, replace=False)
customers.loc[ix_dirty, 'name'] = customers.loc[ix_dirty, 'name'].str.upper()
customers.loc[ix_dirty[:9], 'name'] = '  ' + customers.loc[ix_dirty[:9], 'name'] + '   '

# Duplicate customer_id rows
dup_rows = customers.sample(3, random_state=7).copy()
dup_rows['name'] = dup_rows['name'].str.replace('Client', 'CLIENT', regex=False)
customers = pd.concat([customers, dup_rows], ignore_index=True)

customers.head()

,customer_id,name,age,city,segment
0,1000,Client_001,33.0,Barcelona,Plus
1,1001,CLIENT_002,45.0,None,Premium
2,1002,Client_003,42.0,Bilbao,Basic
3,1003,CLIENT_004,35.0,Valencia,Basic
4,1004,Client_005,26.0,Madrid,Basic


In [4]:
# --- Products ---
product_ids = np.arange(200, 240)
categories = np.random.choice(['Electronics', 'Home', 'Sports', 'Books'], size=len(product_ids), p=[0.35, 0.30, 0.20, 0.15])
base_prices = np.random.uniform(8, 450, size=len(product_ids)).round(2)

products = pd.DataFrame({
    'product_id': product_ids,
    'product_name': [f"Product-{pid}" for pid in product_ids],
    'category': categories,
    'base_price': base_prices
})

# Dirty category labels
products.loc[np.random.choice(products.index, size=6, replace=False), 'category'] = ' electronics '
products.loc[np.random.choice(products.index, size=5, replace=False), 'category'] = 'HOME'

# Duplicate product_id rows
dup_products = products.sample(2, random_state=9).copy()
dup_products['base_price'] = (dup_products['base_price'] * 1.03).round(2)
products = pd.concat([products, dup_products], ignore_index=True)

products.head()

,product_id,product_name,category,base_price
0,200,Product-200,Sports,184.93
1,201,Product-201,Electronics,316.37
2,202,Product-202,Sports,87.59
3,203,Product-203,Books,315.85
4,204,Product-204,electronics,189.95


In [5]:
# --- Transactions ---
n_tx = 900
tx_ids = np.arange(5000, 5000 + n_tx)

tx_customer = np.random.choice(customers['customer_id'].unique(), size=n_tx, replace=True)
tx_product = np.random.choice(list(products['product_id'].unique()) + [np.nan], size=n_tx, replace=True)

quantities = np.random.choice([0, 1, 1, 1, 2, 2, 3, 4], size=n_tx)
discounts = np.random.choice([0.0, 0.0, 0.05, 0.10, 0.15, np.nan], size=n_tx, p=[0.35, 0.15, 0.18, 0.14, 0.10, 0.08])

date_range = pd.date_range('2025-09-01', '2025-12-31', freq='D')
tx_dates = np.random.choice(date_range, size=n_tx)

transactions = pd.DataFrame({
    'tx_id': tx_ids,
    'customer_id': tx_customer,
    'product_id': tx_product,
    'quantity': quantities,
    'discount': discounts,
    'tx_date': pd.to_datetime(tx_dates)
})

# Duplicate tx_id rows
dup_tx = transactions.sample(4, random_state=11).copy()
transactions = pd.concat([transactions, dup_tx], ignore_index=True)

transactions.head()

,tx_id,customer_id,product_id,quantity,discount,tx_date
0,5000,1007,222.0,4,0.15,2025-09-27
1,5001,1025,209.0,4,0.00,2025-10-26
2,5002,1050,231.0,4,0.00,2025-12-15
3,5003,1044,229.0,1,0.00,2025-09-23
4,5004,1043,238.0,1,0.00,2025-09-25


---
## 2. Diagnóstico inicial

Antes de limpiar, siempre se inspecciona:
- tamaño (`shape`)
- tipos y nulos (`info`)
- nulos por columna (`isna().sum()`)
- duplicados (globales y por clave)


### Ejemplo: forma + nulos por columna

In [6]:
print('customers:', customers.shape)
print('products:', products.shape)
print('transactions:', transactions.shape)

display(customers.isna().sum())
display(products.isna().sum())
display(transactions.isna().sum())

customers: (123, 5)
products: (42, 4)
transactions: (904, 6)


customer_id     0
name            0
age            10
city            7
segment         0
dtype: int64

product_id      0
product_name    0
category        0
base_price      0
dtype: int64

tx_id           0
customer_id     0
product_id     25
quantity        0
discount       70
tx_date         0
dtype: int64

### Ejercicio 1 (diagnóstico)

1) Ejecuta `info()` para las tres tablas.
2) Calcula duplicados:
- duplicados completos en cada tabla
- duplicados por clave: `customer_id`, `product_id`, `tx_id`
3) Resume (en una frase) los 3 problemas más importantes que hay que corregir.


In [7]:
customers.info()
products.info()
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  123 non-null    int64  
 1   name         123 non-null    object 
 2   age          113 non-null    float64
 3   city         116 non-null    object 
 4   segment      123 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 4.9+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    42 non-null     int64  
 1   product_name  42 non-null     object 
 2   category      42 non-null     object 
 3   base_price    42 non-null     float64
dtypes: float64(1), int64(1), object(2)
memory usage: 1.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 904 entries, 0 to 903
Data columns (total 6 columns):
 #   Column   

In [8]:
print('Duplicated rows customers:', customers.duplicated().sum())
print('Duplicated rows products:', products.duplicated().sum())
print('Duplicated rows transactions:', transactions.duplicated().sum())

print('Duplicate customer_id:', customers['customer_id'].duplicated().sum())
print('Duplicate product_id:', products['product_id'].duplicated().sum())
print('Duplicate tx_id:', transactions['tx_id'].duplicated().sum())

Duplicated rows customers: 0
Duplicated rows products: 0
Duplicated rows transactions: 4
Duplicate customer_id: 3
Duplicate product_id: 2
Duplicate tx_id: 4


In [9]:
display(customers[customers['customer_id'].duplicated(keep=False)].sort_values('customer_id'))
display(products[products['product_id'].duplicated(keep=False)].sort_values('product_id'))
display(transactions[transactions['tx_id'].duplicated(keep=False)].sort_values('tx_id'))


,customer_id,name,age,city,segment
74,1074,Client_075,41.0,Sevilla,Basic
121,1074,CLIENT_075,41.0,Sevilla,Basic
79,1079,Client_080,24.0,Madrid,Plus
122,1079,CLIENT_080,24.0,Madrid,Plus
112,1112,Client_113,47.0,Zaragoza,Plus
120,1112,CLIENT_113,47.0,Zaragoza,Plus


,product_id,product_name,category,base_price
20,220,Product-220,Home,150.14
40,220,Product-220,Home,154.64
36,236,Product-236,electronics,242.82
41,236,Product-236,electronics,250.10


,tx_id,customer_id,product_id,quantity,discount,tx_date
232,5232,1085,205.0,0,0.10,2025-11-17
902,5232,1085,205.0,0,0.10,2025-11-17
349,5349,1000,213.0,1,0.00,2025-12-03
901,5349,1000,213.0,1,0.00,2025-12-03
398,5398,1115,205.0,1,0.10,2025-12-19
900,5398,1115,205.0,1,0.10,2025-12-19
843,5843,1089,200.0,0,0.05,2025-12-23
903,5843,1089,200.0,0,0.05,2025-12-23


---
## 3. Duplicados

Casos típicos:
- duplicados exactos (mismas columnas)
- duplicados por clave (mismo ID con información repetida o ligeramente distinta)

En dimensiones (`customers`, `products`): normalmente se busca **1 fila por ID**.
En hechos (`transactions`): duplicados por `tx_id` suelen eliminarse.


### Ejemplo: eliminar duplicados completos

In [10]:
transactions_nodup = transactions.drop_duplicates()
print('Before:', transactions.shape)
print('After :', transactions_nodup.shape)

Before: (904, 6)
After : (900, 6)


### Ejemplo: deduplicar por clave

Manteniendo la primera aparición por ID:
- `drop_duplicates(subset=[...], keep='first')`


In [11]:
customers_dedup = customers.drop_duplicates(subset=['customer_id'], keep='first')
products_dedup = products.drop_duplicates(subset=['product_id'], keep='first')
transactions_dedup = transactions.drop_duplicates(subset=['tx_id'], keep='first')

print(customers.shape, '->', customers_dedup.shape)
print(products.shape, '->', products_dedup.shape)
print(transactions.shape, '->', transactions_dedup.shape)

(123, 5) -> (120, 5)
(42, 4) -> (40, 4)
(904, 6) -> (900, 6)


### Ejercicio 2 (unicidad de claves)

Verifica que tras deduplicar:
- `customers_dedup` tiene `customer_id` único
- `products_dedup` tiene `product_id` único
- `transactions_dedup` tiene `tx_id` único

Entrega:
- para cada tabla, compara `nunique()` de la clave con el número de filas.


In [12]:
customers_dedup = customers.drop_duplicates(subset=['customer_id'], keep='first')
products_dedup = products.drop_duplicates(subset=['product_id'], keep='first')
transactions_dedup = transactions.drop_duplicates(subset=['tx_id'], keep='first')

print(customers_dedup['customer_id'].nunique(), customers_dedup.shape[0])
print(products_dedup['product_id'].nunique(), products_dedup.shape[0])
print(transactions_dedup['tx_id'].nunique(), transactions_dedup.shape[0])

120 120
40 40
900 900


---
## 4. Nulos: imputación y eliminación

Estrategias habituales:
- imputación con estadística (media/mediana/moda)
- imputación con categoría explícita (`Unknown`)
- eliminar filas cuando falta una clave crítica

Importante: la estrategia debe ser coherente con el significado del dato.


### Ejemplo: limpiar `customers`

In [13]:
print(customers.isna().sum())
print(products.isna().sum())
print(transactions.isna().sum())

customer_id     0
name            0
age            10
city            7
segment         0
dtype: int64
product_id      0
product_name    0
category        0
base_price      0
dtype: int64
tx_id           0
customer_id     0
product_id     25
quantity        0
discount       70
tx_date         0
dtype: int64


In [14]:
customers_clean = customers_dedup.copy() #We use .copy() to avoid SettingWithCopyWarning when we modify customers_clean
customers_clean['age'] = customers_clean['age'].fillna(customers_clean['age'].median())
customers_clean['city'] = customers_clean['city'].fillna('Unknown')

customers_clean.isna().sum()

customer_id    0
name           0
age            0
city           0
segment        0
dtype: int64

### Ejercicio 3 (nulos en transacciones)

Construye `transactions_clean` a partir de `transactions_dedup`:
- rellena `discount` con 0.0
- elimina filas con `product_id` nulo
- decide qué hacer con `quantity == 0` (elimina o transforma)

Entrega:
- `shape` antes y después
- nulos por columna al final


In [15]:
transactions_clean = transactions_dedup.copy()

transactions_clean['discount'] = transactions_clean['discount'].fillna(0.0)
transactions_clean = transactions_clean.dropna(subset=['product_id'])
# transactions_clean['quantity'] = transactions_clean['quantity'].replace(0, 1) # We replace 0 with the minimum quantity of 1
transactions_clean = transactions_clean[transactions_clean['quantity'] > 0] # We drop transactions with quantity 0, as they don't make sense in our context

transactions_clean.isna().sum()

tx_id          0
customer_id    0
product_id     0
quantity       0
discount       0
tx_date        0
dtype: int64

### Ejemplo: imputación por grupo (cuando tiene sentido)

Ejemplo típico: imputar `age` con mediana por segmento.


In [16]:
customers_filled_age = customers_dedup.copy()
customers_filled_age['age'] = customers_filled_age['age'].fillna(customers_filled_age['age'].median())
customers_filled_age.isna().sum()

customer_id    0
name           0
age            0
city           7
segment        0
dtype: int64

In [17]:
customers_group_impute = customers_dedup.copy()
segment_median_age = customers_group_impute.groupby('segment')['age'].transform('median')
customers_group_impute['age'] = customers_group_impute['age'].fillna(segment_median_age)
customers_group_impute.isna().sum()

customer_id    0
name           0
age            0
city           7
segment        0
dtype: int64

### Ejercicio 4 (comparación de estrategias)

1) Crea `customers_clean_global` (imputación global con mediana).
2) Crea `customers_clean_group` (imputación por segmento).
3) Compara la distribución de `age` en ambos casos (`describe`).

Entrega:
- tabla/resumen que te permita argumentar si hay diferencia relevante.


In [18]:
customers_clean_global = customers_dedup.copy()
customers_clean_global['age'] = customers_clean_global['age'].fillna(customers_clean_global['age'].median())
customers_clean_group = customers_dedup.copy()
segment_median_age = customers_clean_group.groupby('segment')['age'].transform('median')
customers_clean_group['age'] = customers_clean_group['age'].fillna(segment_median_age)
print(customers_clean_global['age'].describe())
print(customers_clean_group['age'].describe())

count    120.000000
mean      36.766667
std       11.350588
min       18.000000
25%       28.750000
50%       37.000000
75%       44.000000
max       75.000000
Name: age, dtype: float64
count    120.000000
mean      36.733333
std       11.370191
min       18.000000
25%       28.750000
50%       37.500000
75%       44.000000
max       75.000000
Name: age, dtype: float64


---
## 5. Limpieza de texto con `.str`

Operaciones frecuentes:
- `strip()` para espacios
- `lower()` / `title()` para normalizar caja
- `replace()` para corregir patrones


### Ejemplo: normalizar categorías en productos

In [19]:
#display(transactions["quantity"].value_counts())

In [20]:
products_clean = products_dedup.copy()

print('Before:')
display(products_clean['category'].value_counts().head(10))

products_clean['category'] = products_clean['category'].astype(str).str.strip().str.title()

print('After:')
display(products_clean['category'].value_counts().head(10))

Before:


category
Home             14
Sports            6
Electronics       6
 electronics      6
HOME              5
Books             3
Name: count, dtype: int64

After:


category
Home           19
Electronics    12
Sports          6
Books           3
Name: count, dtype: int64

### Ejercicio 5 (limpieza de nombres)

Normaliza `customers_clean['name']`:
- elimina espacios
- normaliza formato (p.ej. `title`)

Entrega:
- muestra 10 valores antes y después.


In [21]:
customers_clean = customers_dedup.copy()
customers_clean['name'] = customers_clean['name'].str.strip().str.title()
print("Original names:")
print(customers_dedup['name'].head(10))
print("Cleaned names:")
print(customers_clean['name'].head(10))

Original names:
0          Client_001 
1       CLIENT_002    
2          Client_003 
3          CLIENT_004 
4          Client_005 
5          Client_006 
6       CLIENT_007    
7          Client_008 
8          Client_009 
9          Client_010 
Name: name, dtype: object
Cleaned names:
0    Client_001
1    Client_002
2    Client_003
3    Client_004
4    Client_005
5    Client_006
6    Client_007
7    Client_008
8    Client_009
9    Client_010
Name: name, dtype: object


### Ejercicio 6 (control de categorías)

Asegura que `products_clean['category']` queda dentro de:
{Electronics, Home, Sports, Books}.

Entrega:
- lista de categorías finales
- si hay alguna fuera, decide si la corriges o la filtras.


In [22]:
products_clean = products_dedup.copy()
products_clean['category'] = products_clean['category'].astype(str).str.strip().str.title()

valid_categories = ['Electronics', 'Home', 'Sports', 'Books']
products_clean = products_clean[products_clean['category'].isin(valid_categories)]

products_clean['category'].value_counts()

category
Home           19
Electronics    12
Sports          6
Books           3
Name: count, dtype: int64

---
## 6. `.apply()` (cuándo usarlo)

`.apply()` aplica una función fila a fila o elemento a elemento.

Notas importantes:
- suele ser más lento que operaciones vectorizadas
- se usa cuando la lógica no es fácilmente vectorizable
- para columnas: `df['col'].apply(func)`
- para filas: `df.apply(func, axis=1)`


### Ejemplo 1: `apply` sobre una columna (función de limpieza)

Creamos una función que normaliza nombres con reglas adicionales:
- strip
- title
- reemplazos puntuales


In [23]:
def normalize_name(x: str) -> str:
    if pd.isna(x):
        return x
    x = str(x).strip()
    x = x.replace('_', ' ')
    x = x.title()
    return x

demo = customers_dedup[['name']].copy()
demo['name_norm'] = demo['name'].apply(normalize_name)
display(demo.head(10))

,name,name_norm
0,Client_001,Client 001
1,CLIENT_002,Client 002
2,Client_003,Client 003
3,CLIENT_004,Client 004
4,Client_005,Client 005
5,Client_006,Client 006
6,CLIENT_007,Client 007
7,Client_008,Client 008
8,Client_009,Client 009
9,Client_010,Client 010


### Ejemplo 2: `apply` por filas (`axis=1`)

Creamos un label con lógica combinada (edad y segmento) para ilustrar `axis=1`.

In [24]:
def customer_label(row) -> str:
    age = row['age']
    seg = row['segment']
    if age >= 50 and seg == 'Premium':
        return 'VIP_Senior'
    if age >= 50:
        return 'Senior'
    if seg == 'Premium':
        return 'Premium'
    return 'Standard'

demo2 = customers_clean[['age', 'segment']].copy()
demo2['label'] = demo2.apply(customer_label, axis=1)
demo2['label'].value_counts()

label
Standard      92
Premium       16
Senior        11
VIP_Senior     1
Name: count, dtype: int64

### Ejercicio 7 (`apply`)

1) Define una función `discount_band(d)` que devuelva:
- `'none'` si `d == 0`
- `'low'` si `0 < d < 0.10`
- `'high'` si `d >= 0.10`

2) Crea en `transactions_clean` una columna `discount_band` usando `apply`.
3) Muestra `value_counts()` de `discount_band`.


In [25]:
def discount_band(d):
    if d == 0:
        return 'none'
    elif d < 0.10:
        return 'low'
    else:
        return 'high'

transactions_clean['discount_band'] = transactions_clean['discount'].apply(discount_band)

transactions_clean['discount_band'].value_counts()

discount_band
none    419
high    196
low     152
Name: count, dtype: int64

---
## 7. Tipos (dtypes) y validaciones

Reglas prácticas:
- IDs no deberían ser float.
- fechas deben ser datetime.
- precios y cantidades deben tener dominios válidos.

Validaciones con `assert` ayudan a detectar errores pronto.


### Ejemplo: asserts mínimos

In [26]:
assert customers_clean['customer_id'].isna().sum() == 0
assert products_clean['product_id'].isna().sum() == 0
assert (products_clean['base_price'] > 0).all()
print('Basic checks passed')

Basic checks passed


### Ejercicio 8 (dtypes + asserts)

1) Asegura que `transactions_clean['product_id']` es entero (o al menos no float con decimales).
2) Asegura que `tx_date` es datetime.
3) Añade al menos 6 `assert` relevantes, por ejemplo:
- `discount` en [0, 1]
- `quantity` >= 1
- `amount` no negativo (cuando lo crees)
- IDs sin nulos

Entrega:
- imprime `dtypes` de tablas limpias.


In [27]:
transactions_clean['product_id'] = transactions_clean['product_id'].astype(int)
transactions_clean['customer_id'] = transactions_clean['customer_id'].astype(int)

assert transactions_clean['discount'].between(0, 1).all()
assert (transactions_clean['quantity'] >= 1).all()
assert (products_clean['base_price'] > 0).all()
assert customers_clean['customer_id'].isna().sum() == 0

print('Validations passed')
print(transactions_clean.dtypes)

Validations passed
tx_id                     int64
customer_id               int64
product_id                int64
quantity                  int64
discount                float64
tx_date          datetime64[ns]
discount_band            object
dtype: object


---
## 8. `merge`: uniones para construir la tabla analítica

Conceptos:
- clave de unión (IDs)
- tipo de unión (`how`): `left` vs `inner`
- nulos post-merge (cuando no hay match)

Patrón habitual:
- unir hechos (`transactions_clean`) con dimensiones (`customers_clean`, `products_clean`) con `left`.

### Ejemplo: `left` vs `inner` en un caso mínimo

In [28]:
demo_left = pd.DataFrame({'id': [1, 2, 3], 'x': [10, 20, 30]})
demo_right = pd.DataFrame({'id': [2, 3, 4], 'y': [200, 300, 400]})

inner_join = pd.merge(demo_left, demo_right, on='id', how='inner')
left_join = pd.merge(demo_left, demo_right, on='id', how='left')

display(inner_join)
display(left_join)

,id,x,y
0,2,20,200
1,3,30,300


,id,x,y
0,1,10,NaN
1,2,20,200.0
2,3,30,300.0


### Ejercicio 9 (tipos de unión)

Construye:
- `right_join`
- `outer_join`

Entrega:
- número de filas de cada join
- qué join introduce más nulos y por qué


In [34]:
right_join = pd.merge(demo_left, demo_right, on='id', how='right')
outer_join = pd.merge(demo_left, demo_right, on='id', how='outer')

display(right_join)
display(outer_join)

,id,x,y
0,2,20.0,200
1,3,30.0,300
2,4,NaN,400


,id,x,y
0,1,10.0,NaN
1,2,20.0,200.0
2,3,30.0,300.0
3,4,NaN,400.0


---
## 9. Construcción de `fact_sales`

`fact_sales` será una tabla a nivel transacción con:
- columnas de transacción (`tx_id`, `quantity`, `discount`, `tx_date`)
- columnas de producto (`category`, `base_price`, `product_name`)
- columnas de cliente (`city`, `segment`, `age`, `name`)
- métrica monetaria `amount`


### Ejemplo: cálculo de `amount`

`amount = quantity * base_price * (1 - discount)`


In [30]:
example_amount = (2 * 100.0) * (1 - 0.10)
example_amount

180.0

In [37]:
display(transactions_clean[transactions_clean['discount'] < 0])

,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band


### Ejercicio 10 (construir `fact_sales`)

1) Une `transactions_clean` con `products_clean` por `product_id` (`how='left'`).
2) Une el resultado con `customers_clean` por `customer_id` (`how='left'`).
3) Crea la columna `amount`.

Entrega:
- `shape`
- `amount.describe()`
- nulos por columna en campos críticos (`base_price`, `segment`, `city`, `amount`).


In [38]:
fact_sales = pd.merge(
    transactions_clean,
    products_clean,
    on='product_id',
    how='left'
)

fact_sales = pd.merge(
    fact_sales,
    customers_clean,
    on='customer_id',
    how='left'
)

fact_sales['amount'] = (
    fact_sales['quantity']
    * fact_sales['base_price']
    * (1 - fact_sales['discount'])
)

fact_sales[['amount']].describe()

,amount
count,767.000000
mean,433.602018
std,367.723355
min,7.777500
25%,161.457500
50%,320.820000
75%,631.700000
max,1775.720000


### Ejercicio 11 (calidad post-merge)

1) Identifica filas de `fact_sales` donde falten datos de dimensión (por ejemplo `base_price` nulo).
2) Calcula cuántas son y qué porcentaje representan.
3) Decide una estrategia: filtrar o imputar.

Entrega:
- una celda con tu decisión aplicada (no hace falta que sea la única válida).


In [40]:
display(fact_sales[fact_sales["base_price"].isna()].head(5))
## Número de NaN

print(fact_sales[fact_sales["base_price"].isna()].shape[0])

## Número total de filas
print(fact_sales.shape[0])

## Porcentaje de nulos
print(fact_sales[fact_sales["base_price"].isna()].shape[0]/fact_sales.shape[0])

## Como son pocos podríamos hacer un dropna 
fact_sales_final = fact_sales.dropna(subset = ["base_price"])

## Vemos cómo queda
fact_sales_final.head(5)

,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band,product_name,category,base_price,name,age,city,segment,amount


0
767
0.0


,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band,product_name,category,base_price,name,age,city,segment,amount
0,5000,1007,222,4,0.15,2025-09-27,high,Product-222,Books,220.67,Client_008,46.0,Zaragoza,Premium,750.278
1,5001,1025,209,4,0.00,2025-10-26,none,Product-209,Home,106.94,Client_026,75.0,Bilbao,Basic,427.760
2,5002,1050,231,4,0.00,2025-12-15,none,Product-231,Electronics,443.93,Client_051,62.0,None,Plus,1775.720
3,5003,1044,229,1,0.00,2025-09-23,none,Product-229,Electronics,45.36,Client_045,NaN,Madrid,Basic,45.360
4,5004,1043,238,1,0.00,2025-09-25,none,Product-238,Home,35.55,Client_044,23.0,Zaragoza,Basic,35.550


In [42]:
display(fact_sales_final.sample(5, random_state=42))

,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band,product_name,category,base_price,name,age,city,segment,amount
667,5777,1106,227,4,0.00,2025-12-10,none,Product-227,Home,9.15,Client_107,34.0,Barcelona,Premium,36.600
324,5376,1058,229,4,0.15,2025-11-15,high,Product-229,Electronics,45.36,Client_059,42.0,Madrid,Premium,154.224
623,5727,1013,227,1,0.00,2025-10-18,none,Product-227,Home,9.15,Client_014,38.0,Madrid,Basic,9.150
689,5807,1101,225,3,0.00,2025-11-19,none,Product-225,Sports,77.38,Client_102,18.0,Sevilla,Plus,232.140
521,5608,1100,234,1,0.10,2025-11-23,high,Product-234,Electronics,112.36,Client_101,62.0,Madrid,Plus,101.124


---
## 10. Features mínimas listas para análisis

En esta sesión nos quedamos en features simples para poder analizar después:
- `month` y `weekday` desde `tx_date`
- `age_group` desde `age`

En Pandas III se explotará esto en reporting y análisis.


### Ejemplo: extraer `month` y `weekday`

In [46]:
tmp = fact_sales_final.copy()
tmp['month'] = tmp['tx_date'].dt.month
tmp['weekday'] = tmp['tx_date'].dt.day_name()
display(tmp[['tx_date', 'month', 'weekday']].head())

,tx_date,month,weekday
0,2025-09-27,9,Saturday
1,2025-10-26,10,Sunday
2,2025-12-15,12,Monday
3,2025-09-23,9,Tuesday
4,2025-09-25,9,Thursday


### Ejercicio 12 (features en `fact_sales`)

Crea en `fact_sales`:
- `month`
- `weekday`
- `age_group` con `pd.cut` (Young/Adult/Senior)

Entrega:
- `value_counts()` de `weekday` y `age_group`.


In [62]:
fact_sales_final['month'] = fact_sales_final['tx_date'].dt.month
fact_sales_final['weekday'] = fact_sales_final['tx_date'].dt.day_name()

display(fact_sales_final.sample(3))
display(fact_sales_final[fact_sales_final['age'] > 65])

bins = [0, 29, 49, 200]
labels = ['Young', 'Adult', 'Senior']

fact_sales_final['age_group'] = pd.cut(fact_sales_final['age'], bins=bins, labels=labels)
display(fact_sales_final[['age', 'age_group']].sample(10))


,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band,product_name,category,base_price,name,age,city,segment,amount,month,weekday,age_group
28,5034,1022,214,1,0.0,2025-12-18,none,Product-214,Books,250.53,Client_023,42.0,Barcelona,Basic,250.53,12,Thursday,Adult
313,5365,1031,200,3,0.0,2025-11-18,none,Product-200,Sports,184.93,Client_032,45.0,Madrid,Basic,554.79,11,Tuesday,Adult
532,5621,1108,215,1,0.0,2025-12-19,none,Product-215,Home,244.22,Client_109,29.0,Madrid,Premium,244.22,12,Friday,Young


,tx_id,customer_id,product_id,quantity,discount,tx_date,discount_band,product_name,category,base_price,name,age,city,segment,amount,month,weekday,age_group
1,5001,1025,209,4,0.00,2025-10-26,none,Product-209,Home,106.94,Client_026,75.0,Bilbao,Basic,427.7600,10,Sunday,Senior
7,5007,1025,235,3,0.00,2025-11-02,none,Product-235,Home,343.90,Client_026,75.0,Bilbao,Basic,1031.7000,11,Sunday,Senior
80,5091,1025,216,1,0.00,2025-12-15,none,Product-216,Home,165.35,Client_026,75.0,Bilbao,Basic,165.3500,12,Monday,Senior
400,5471,1025,214,3,0.00,2025-09-19,none,Product-214,Books,250.53,Client_026,75.0,Bilbao,Basic,751.5900,9,Friday,Senior
435,5511,1025,200,1,0.15,2025-10-24,high,Product-200,Sports,184.93,Client_026,75.0,Bilbao,Basic,157.1905,10,Friday,Senior
448,5525,1025,215,1,0.10,2025-11-13,high,Product-215,Home,244.22,Client_026,75.0,Bilbao,Basic,219.7980,11,Thursday,Senior
480,5560,1025,201,4,0.00,2025-09-28,none,Product-201,Electronics,316.37,Client_026,75.0,Bilbao,Basic,1265.4800,9,Sunday,Senior
531,5619,1025,215,1,0.05,2025-12-26,low,Product-215,Home,244.22,Client_026,75.0,Bilbao,Basic,232.0090,12,Friday,Senior
533,5622,1025,211,1,0.10,2025-09-20,high,Product-211,Sports,160.53,Client_026,75.0,Bilbao,Basic,144.4770,9,Saturday,Senior
540,5630,1025,229,2,0.00,2025-10-12,none,Product-229,Electronics,45.36,Client_026,75.0,Bilbao,Basic,90.7200,10,Sunday,Senior


,age,age_group
93,62.0,Senior
38,40.0,Adult
766,32.0,Adult
393,42.0,Adult
696,62.0,Senior
277,47.0,Adult
552,42.0,Adult
516,46.0,Adult
31,45.0,Adult
522,29.0,Young


---
## 11. Ejercicios extra (si sobra tiempo)

Estos ejercicios refuerzan lo anterior y dejan el dataset muy listo para Pandas III.

### Ejercicio 13 (control de dominios)

Valida dominios en `fact_sales`:
- `discount` en [0, 1]
- `quantity` >= 1
- `base_price` > 0
- `amount` >= 0

Entrega:
- asserts y una celda que pase.


In [63]:
assert (fact_sales_final['discount'] >= 0).all() and (fact_sales_final['discount'] <= 1).all(), "discount fuera de rango [0, 1]"
assert (fact_sales_final['quantity'] >= 1).all(), "quantity menor a 1"
assert (fact_sales_final['base_price'] > 0).all(), "base_price no positivo"
assert (fact_sales_final['amount'] >= 0).all(), "amount negativo"

print("✓ Todos los dominios son válidos")

✓ Todos los dominios son válidos


### Ejercicio 14 (trazabilidad)

Crea una tabla `data_quality` que resuma para cada DataFrame limpio:
- filas
- columnas
- nulos totales
- número de IDs únicos (según la clave)

Entrega:
- `data_quality` como DataFrame.


In [64]:
data_quality = pd.DataFrame({
    'table': ['customers_clean', 'products_clean', 'transactions_clean', 'fact_sales_final'],
    'rows': [
        customers_clean.shape[0],
        products_clean.shape[0],
        transactions_clean.shape[0],
        fact_sales_final.shape[0]
    ],
    'columns': [
        customers_clean.shape[1],
        products_clean.shape[1],
        transactions_clean.shape[1],
        fact_sales_final.shape[1]
    ],
    'total_nulls': [
        customers_clean.isna().sum().sum(),
        products_clean.isna().sum().sum(),
        transactions_clean.isna().sum().sum(),
        fact_sales_final.isna().sum().sum()
    ],
    'unique_ids': [
        customers_clean['customer_id'].nunique(),
        products_clean['product_id'].nunique(),
        transactions_clean['tx_id'].nunique(),
        fact_sales_final['tx_id'].nunique()
    ]
})

display(data_quality)

,table,rows,columns,total_nulls,unique_ids
0,customers_clean,120,5,17,120
1,products_clean,40,4,0,40
2,transactions_clean,767,7,0,767
3,fact_sales_final,767,18,179,767
